# 04 · how machines see

**before you start:** *Runtime → Change runtime type → T4 GPU → Save.* it's free, and it makes the AI models in this notebook about 20 times faster.

**how to use this:** press ▶ on each cell, top to bottom. anything in a box on the right is yours to change. change it, press ▶ again, see what happens. you can't break anything. if it gets weird: *Runtime → Restart session* and start from the top.

to a computer, a photo is **just a big table of numbers**. everything in this notebook is maths on that table.

every image box below takes a **web link to a picture**, or type `upload` to use your own photo.

In [ ]:
#@title ▶ run me first (loads the tools, takes a few seconds)
import io, requests, numpy as np, cv2, matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline
import warnings; warnings.filterwarnings("ignore")

def load_image(url_or_upload):
    """a web link, or 'upload' to pick a file from your computer"""
    if url_or_upload == "upload":
        from google.colab import files
        f = files.upload(); return Image.open(io.BytesIO(next(iter(f.values())))).convert("RGB")
    r = requests.get(url_or_upload, headers={"User-Agent": "Mozilla/5.0 (ai-bootcamp notebook)"}, timeout=30)
    r.raise_for_status()
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    img.thumbnail((1024, 1024))
    return img

def show(img, title="", size=6):
    plt.figure(figsize=(size, size)); plt.imshow(img, cmap="gray" if np.ndim(img) == 2 else None); plt.axis("off"); plt.title(title); plt.show()

print("ready ✅")

## 1 · a photo is numbers

In [ ]:
#@title load a photo
image_link = "https://commons.wikimedia.org/wiki/Special:FilePath/Golden_Retriever_English_cream_portrait.jpg?width=1024" #@param {type:"string"}
img = load_image(image_link)
show(img, f"{img.width} x {img.height} pixels")

In [ ]:
#@title zoom right in
zoom = 12 #@param {type:"slider", min:6, max:20, step:1}
gray = np.array(img.convert("L"))
# find the most detailed spot in the photo (where the numbers change the most), then zoom in there
step = max(zoom, 8)
spots = [(gray[r:r + zoom, c:c + zoom].std(), r, c) for r in range(0, gray.shape[0] - zoom, step) for c in range(0, gray.shape[1] - zoom, step)]
_, r0, c0 = max(spots)
patch = gray[r0:r0 + zoom, c0:c0 + zoom]
plt.figure(figsize=(9, 9)); plt.imshow(patch, cmap="gray", vmin=0, vmax=255)
for (r, c), v in np.ndenumerate(patch):
    plt.text(c, r, v, ha="center", va="center", fontsize=8, color="red" if v > 127 else "yellow")
plt.axis("off"); plt.title(f"{zoom} x {zoom} pixels from the busiest bit of the photo. 0 = black, 255 = white"); plt.show()
print(f"the whole photo is {gray.shape[0] * gray.shape[1]:,} of these numbers. three times over, for red, green and blue.")

In [ ]:
#@title split into red, green and blue
rgb = np.array(img)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, i, name, cmap in zip(axes, range(3), ["red", "green", "blue"], ["Reds", "Greens", "Blues"]):
    ax.imshow(rgb[:, :, i], cmap=cmap); ax.set_title(name); ax.axis("off")
plt.show()

## 2 · photo effects are just maths

In [ ]:
#@title apply an effect
effect = "edges" #@param ["grayscale", "edges", "blur", "cartoon", "pencil sketch", "invert"]
strength = 100 #@param {type:"slider", min:10, max:250, step:10}
bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
if effect == "grayscale":
    out = gray
elif effect == "edges":
    out = cv2.Canny(gray, strength // 2, strength)
elif effect == "blur":
    k = strength // 10 * 2 + 1; out = cv2.cvtColor(cv2.GaussianBlur(bgr, (k, k), 0), cv2.COLOR_BGR2RGB)
elif effect == "cartoon":
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray, 7), 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 9)
    colour = cv2.bilateralFilter(bgr, 9, strength, strength)
    out = cv2.cvtColor(cv2.bitwise_and(colour, colour, mask=edges), cv2.COLOR_BGR2RGB)
elif effect == "pencil sketch":
    out = cv2.divide(gray, 255 - cv2.GaussianBlur(255 - gray, (21, 21), 0), scale=256)
else:
    out = 255 - np.array(img)
show(out, effect, size=7)

**edges** is the interesting one: the computer finds every place where the numbers jump suddenly. that's roughly what the first layer of a vision network learns to do by itself.

## 3 · what's in this photo?
a neural network that has already seen **millions of photos** in 1,000 categories. you just ask it.

In [ ]:
#@title recognise a photo
image_link = "https://commons.wikimedia.org/wiki/Special:FilePath/Golden_Retriever_English_cream_portrait.jpg?width=1024" #@param {type:"string"}
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
img = load_image(image_link)
show(img, size=5)
for guess in classifier(img, top_k=5):
    print(f"{guess['score']:6.1%}  {guess['label']}")

try a photo of something weird. it only knows its 1,000 categories, so it will **confidently pick the closest one**, even when it's wrong.

## 4 · find every object

In [ ]:
#@title draw boxes around things
image_link = "https://commons.wikimedia.org/wiki/Special:FilePath/Busy_busy_Brigade_Road_in_Bangalore_(5730298060).jpg?width=1280" #@param {type:"string"}
how_sure = 0.9 #@param {type:"slider", min:0.3, max:0.99, step:0.01}
detector = pipeline("object-detection", model="hustvl/yolos-tiny")
img = load_image(image_link)
found = [f for f in detector(img) if f["score"] >= how_sure]
plt.figure(figsize=(12, 8)); plt.imshow(img); ax = plt.gca()
for f in found:
    b = f["box"]
    ax.add_patch(plt.Rectangle((b["xmin"], b["ymin"]), b["xmax"] - b["xmin"], b["ymax"] - b["ymin"], fill=False, color="yellow", linewidth=3))
    ax.text(b["xmin"], b["ymin"] - 6, f"{f['label']} {f['score']:.0%}", color="black", backgroundcolor="yellow", fontsize=10)
plt.axis("off"); plt.title(f"found {len(found)} things"); plt.show()

brigade road, bangalore. lower `how_sure` and it finds more, including things that aren't there. self-driving cars live on this slider.

## 5 · recognise anything you can describe
no training at all. **you write the labels**, it picks the one that matches the photo best.

In [ ]:
#@title your own labels
image_link = "https://images.pexels.com/photos/12737817/pexels-photo-12737817.jpeg?w=1024" #@param {type:"string"}
labels = "biryani, pizza, dosa, a salad, a cake" #@param {type:"string"}
matcher = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")
img = load_image(image_link)
show(img, size=5)
for guess in matcher(img, candidate_labels=[l.strip() for l in labels.split(",")]):
    print(f"{guess['score']:6.1%}  {guess['label']}")

this one is the base for the final project: *ripe or unripe banana*, *healthy or sick plant*, *my handwriting or my friend's*. change the labels and see how far you can push it.

**be suspicious of it.** it always picks *something*, even if none of your labels are right.

<small>photos: golden retriever by Christian Madden (CC BY-SA 2.0) · Brigade Road by Ryan from Toronto (CC BY 2.0) · biryani by dhiraj jain (Pexels). all via Wikimedia Commons / Pexels.</small>